# PARKLENS — Member B Notebook
## Mappls API + MBPR Impact Engine + VRP Deployment

**What this notebook delivers:**
- `mappls_cache.json` — POI demand, Snap-to-Road lane/road-class, ETA calibration per hotspot
- `hotspots_enriched.csv` — hotspots.csv + `V_calibrated`, `delay_ratio`, `veh_min_lost`, `rupees_lost`, `poi_demand` (Contract 2 final fields)
- `routing_plan.json` — fair VRP patrol deployment per police station
- `impact_bpr.py` + `mappls_client.py` — importable modules for Member C's twin

**Run order:** Cell by cell top to bottom. All Mappls calls are cached — re-running is free after the first run.

**You need:**
1. Your Mappls REST API key (get from https://apis.mappls.com/console/)
2. `hotspots.csv` and `clean_violations.parquet` uploaded to `/content/data/`

In [ ]:
# ── CELL 1: Install dependencies ─────────────────────────────────────────────
!pip install -q pyarrow h3 requests pandas numpy scipy tqdm polyline

In [ ]:
# ── CELL 2: Config — SET YOUR KEY HERE ───────────────────────────────────────
import os

# Paths
DATA_DIR   = "/content/data"
OUT_DIR    = "/content/output"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR,  exist_ok=True)

# MBPR constants (Anwar, Fujiwara & Zhang 2011 — Dhaka urban arterial)
ALPHA = 3.59
BETA  = 0.40

# Value of time (India urban, RBI 2023 approx)
VOT_RS_PER_MIN = 3.0   # ₹/vehicle-minute

# Per-lane saturation capacity (urban arterial, veh/h)
SAT_FLOW = 1800

# Footprint → capacity-removed fraction (C_io / lane)
FOOTPRINT_C_IO = {
    0.5:  0.10,   # scooter/motorcycle
    1.0:  0.25,   # car
    1.5:  0.40,   # auto/maxi-cab
    2.0:  0.60,   # LGV
    3.0:  1.00,   # HGV/bus/tanker
}

# Road-class defaults (fallback if Snap-to-Road fails)
ROAD_CLASS_DEFAULTS = {
    "motorway":   {"lanes": 3, "v0_kmh": 80},
    "arterial":   {"lanes": 2, "v0_kmh": 50},
    "collector":  {"lanes": 2, "v0_kmh": 40},
    "local":      {"lanes": 1, "v0_kmh": 30},
    "unknown":    {"lanes": 2, "v0_kmh": 40},
}

# Enforcement window (hours of patrol per day)
ENFORCEMENT_WINDOW_H = 8

print("Config OK")

Config OK


In [ ]:
# ── CELL 3 REPLACEMENT: correct Mappls auth + correct endpoints ───────────────
import json, time, requests
from pathlib import Path

MAPPLS_KEY = ""   # passed as ?access_token=

CACHE_FILE = Path("/content/output/mappls_cache.json")
def _load_cache(): return json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {}
def _save_cache(c): CACHE_FILE.write_text(json.dumps(c, indent=2))
CACHE = _load_cache()

# ── 1. Nearby POI (along-route POST, category filter) ─────────────────────────
# No polyline — we fake a tiny 50m route segment centred on the hotspot point
# and use buffer=300 to get nearby POIs. Categories from battle plan.
import polyline as _poly   # !pip install polyline

def _point_to_tiny_polyline(lat, lon, offset=0.0003):
    """Encode a trivial 2-point path near the hotspot as polyline5."""
    pts = [(lat - offset, lon), (lat + offset, lon)]
    return _poly.encode(pts, 5)

CATEGORIES = ["TRNPMP", "FODCOF", "HOTPRE", "SHPG", "EDUC", "HOSP"]

def get_nearby_poi(lat, lon, zone_id=None):
    key = f"poi_{zone_id or f'{lat:.5f}_{lon:.5f}'}"
    if key in CACHE: return CACHE[key]

    path = _point_to_tiny_polyline(lat, lon)
    total = 0

    for cat in CATEGORIES:
        try:
            r = requests.post(
                f"https://search.mappls.com/search/places/along-route?access_token={MAPPLS_KEY}",
                data={
                    "geometries": "polyline5",
                    "path":       path,
                    "category":   cat,
                    "buffer":     "1000",
                    "page":       "1",
                    "sort":       "",
                },
                timeout=15
            )
            if r.status_code == 200:
                total += len(r.json().get("suggestedPOIs", []))
        except Exception as e:
            print(f"  [POI:{cat}] {e}")
        time.sleep(0.1)

    CACHE[key] = total
    _save_cache(CACHE)
    time.sleep(0.1)
    return total

# ── 2. Snap-to-Road ───────────────────────────────────────────────────────────
# Endpoint: POST https://route.mappls.com/routev2/movement/trace_route
# points = lon,lat format (note: lon first per their curl example)

def get_road_class(lat, lon, zone_id=None):
    key = f"road_{zone_id or f'{lat:.5f}_{lon:.5f}'}"
    if key in CACHE: return CACHE[key]

    result = {"lanes": 2, "v0_kmh": 40, "road_class": "unknown"}
    try:
        # API needs at least 2 points — pass same point offset slightly
        lon2, lat2 = lon + 0.0001, lat + 0.0001
        r = requests.post(
            f"https://route.mappls.com/route/movement/snapToRoad?access_token={MAPPLS_KEY}",
            data={
                "pts":        f"{lon},{lat};{lon2},{lat2}",
                "radiuses":   "50;50",
                "geometries": "polyline",
            },
            timeout=10
        )
        print(f"  [Snap] {r.status_code}: {r.text[:500]}")
    except Exception as e:
        print(f"  [Snap] {e}")

    CACHE[key] = result
    _save_cache(CACHE)
    return result
# ── 3. Predictive Route ETA (for V calibration) ───────────────────────────────
# GET https://route.mappls.com/routev2/direction/route
# locations = lon,lat;lon,lat (lon first)

def get_eta_ratio(lat, lon, zone_id=None):
    key = f"eta_{zone_id or f'{lat:.5f}_{lon:.5f}'}"
    if key in CACHE: return CACHE[key]

    dlat = lat + 0.0045   # ~500m north
    result = {"eta_ratio": 1.0}
    r = requests.get(
        f"https://route.mappls.com/routev2/direction/route",
        params={
            "locations":   f"{lon},{lat};{lon},{dlat}",   # lon,lat per docs
            "profile":     "driving",
            "speedTypes":  "optimal",
            "access_token": MAPPLS_KEY,
        },
        timeout=10
    )
    try:
        d = r.json()
        route   = d["routes"][0]
        typ_s   = route["duration"]       # typical seconds
        dist_m  = route["distance"]
        v0_ms   = 40 / 3.6
        free_s  = dist_m / v0_ms
        ratio   = typ_s / free_s if free_s > 0 else 1.0
        result  = {"eta_ratio": round(ratio, 4)}
    except:
        print(f"  [ETA] {r.status_code}: {r.text[:200]}")

    CACHE[key] = result
    _save_cache(CACHE)
    time.sleep(0.15)
    return result

# ── 4. Route geometry for Member C twin ───────────────────────────────────────
def route_geometry(origin_lat, origin_lon, dest_lat, dest_lon, corridor_id=None):
    key = f"route_{corridor_id or f'{origin_lat:.4f}_{origin_lon:.4f}'}"
    if key in CACHE: return CACHE[key]

    result = {"type": "LineString", "coordinates": []}
    r = requests.get(
        f"https://route.mappls.com/routev2/direction/route",
        params={
            "locations":    f"{origin_lon},{origin_lat};{dest_lon},{dest_lat}",
            "profile":      "driving",
            "speedTypes":   "optimal",
            "geometries":   "geojson",
            "access_token": MAPPLS_KEY,
        },
        timeout=15
    )
    try:
        result = r.json()["routes"][0]["geometry"]
    except:
        print(f"  [Route] {r.status_code}: {r.text[:200]}")

    CACHE[key] = result
    _save_cache(CACHE)
    time.sleep(0.2)
    return result

print(f"Client ready. Cache entries: {len(CACHE)}")

Client ready. Cache entries: 101


In [ ]:
# ── CELL 4: Load Member A outputs ─────────────────────────────────────────────
import pandas as pd
import numpy as np

hotspots = pd.read_csv(f"{DATA_DIR}/hotspots.csv")
print(f"Hotspots: {len(hotspots)} cells")
print(f"Blindspot cells: {hotspots['blindspot_flag'].sum()}")
print(f"Treated (enforcement) cells: {hotspots['is_treated'].sum()}")

# We enrich top-N zones by risk_debiased to stay within Mappls quota
TOP_N = 100
top_zones = hotspots.nlargest(TOP_N, "risk_debiased").reset_index(drop=True)
print(f"\nEnriching top {TOP_N} zones by debiased risk")
print(top_zones[["h3_cell", "lat", "lon", "risk_debiased", "blindspot_flag", "severity_score"]].head(10))

Hotspots: 2534 cells
Blindspot cells: 620
Treated (enforcement) cells: 572

Enriching top 100 zones by debiased risk
           h3_cell        lat        lon  risk_debiased  blindspot_flag  \
0  8960145ac57ffff  12.930695  77.504796       0.010377               1   
1  8961892f463ffff  13.037876  77.734766       0.009054               1   
2  8961892f66bffff  13.063528  77.740034       0.008598               1   
3  8961892f5a7ffff  13.014433  77.710251       0.007400               1   
4  8960145141bffff  12.875033  77.532729       0.007110               1   
5  89618926b8bffff  12.847023  77.588790       0.006843               1   
6  8961892ab8fffff  13.054877  77.766418       0.006227               1   
7  8961892e417ffff  12.974870  77.707762       0.006060               1   
8  8961892f647ffff  13.059233  77.744783       0.005997               1   
9  896189209bbffff  12.926778  77.697898       0.005920               1   

   severity_score  
0        8.000000  
1        7.000000

In [ ]:
CACHE = {k:v for k,v in CACHE.items() if not k.startswith("road_")}
_save_cache(CACHE)
get_road_class(12.930695, 77.504796, zone_id="test")

  [Snap] 200: {"Server":"DE-5402","version":"202404.221.5223","results":{"snappedPoints":[{"distance":4.818717,"location":[77.504821,12.930659],"waypoint_index":0},{"distance":7.990688,"location":[77.504937,12.930735],"waypoint_index":1}]},"responseCode":200}


{'lanes': 2, 'v0_kmh': 40, 'road_class': 'unknown'}

In [ ]:
# ── CELL 5: Fill poi_demand from Mappls Nearby ────────────────────────────────
# This fills the STUB in hotspots.csv.
# ~100 API calls, ~20-30 seconds total.

from tqdm import tqdm

poi_counts = []
for _, row in tqdm(top_zones.iterrows(), total=len(top_zones), desc="POI demand"):
    cnt = get_nearby_poi(row["lat"], row["lon"], zone_id=row["h3_cell"])
    poi_counts.append(cnt)

top_zones["poi_demand"] = poi_counts

# Merge back to all hotspots (non-enriched get 0)
hotspots = hotspots.merge(
    top_zones[["h3_cell", "poi_demand"]].rename(columns={"poi_demand": "poi_demand_new"}),
    on="h3_cell", how="left"
)
hotspots["poi_demand"] = hotspots["poi_demand_new"].fillna(0).astype(int)
hotspots.drop(columns=["poi_demand_new"], inplace=True)

print(f"\nPOI demand filled. Top zones:")
print(top_zones[["h3_cell", "lat", "lon", "risk_debiased", "poi_demand", "blindspot_flag"]].head(10).to_string())

POI demand: 100%|██████████| 100/100 [00:00<00:00, 8012.81it/s]


POI demand filled. Top zones:
           h3_cell        lat        lon  risk_debiased  poi_demand  blindspot_flag
0  8960145ac57ffff  12.930695  77.504796       0.010377          10               1
1  8961892f463ffff  13.037876  77.734766       0.009054           4               1
2  8961892f66bffff  13.063528  77.740034       0.008598           5               1
3  8961892f5a7ffff  13.014433  77.710251       0.007400          13               1
4  8960145141bffff  12.875033  77.532729       0.007110           5               1
5  89618926b8bffff  12.847023  77.588790       0.006843           5               1
6  8961892ab8fffff  13.054877  77.766418       0.006227           2               1
7  8961892e417ffff  12.974870  77.707762       0.006060          14               1
8  8961892f647ffff  13.059233  77.744783       0.005997           5               1
9  896189209bbffff  12.926778  77.697898       0.005920           8               1


In [ ]:
# Step 1 — recompute rc_agg
viol_rc = pd.read_parquet(f"{DATA_DIR}/clean_violations.parquet", columns=["h3_cell", "vehicle_footprint"])
rc_agg  = viol_rc.groupby("h3_cell").agg(
    avg_footprint=("vehicle_footprint", "mean"),
    total_violations=("vehicle_footprint", "count")
).reset_index()

fp_p75 = rc_agg["avg_footprint"].quantile(0.75)
fp_p50 = rc_agg["avg_footprint"].quantile(0.50)
vc_p75 = rc_agg["total_violations"].quantile(0.75)
vc_p50 = rc_agg["total_violations"].quantile(0.50)

def infer_road_class(row):
    fp = row["avg_footprint"]
    vc = row["total_violations"]
    if fp >= fp_p75 or vc >= vc_p75: return {"road_class": "arterial",  "lanes": 2, "v0_kmh": 50}
    if fp >= fp_p50 or vc >= vc_p50: return {"road_class": "collector", "lanes": 2, "v0_kmh": 40}
    return                                   {"road_class": "local",     "lanes": 1, "v0_kmh": 30}

rc_agg["rc_info"]    = rc_agg.apply(infer_road_class, axis=1)
rc_agg["road_class"] = rc_agg["rc_info"].apply(lambda x: x["road_class"])
rc_agg["lanes"]      = rc_agg["rc_info"].apply(lambda x: x["lanes"])
rc_agg["v0_kmh"]     = rc_agg["rc_info"].apply(lambda x: x["v0_kmh"])

# Step 2 — drop old columns from top_zones
for col in ["road_class", "lanes", "v0_kmh", "road_class_x", "road_class_y",
            "lanes_x", "lanes_y", "v0_kmh_x", "v0_kmh_y"]:
    if col in top_zones.columns:
        top_zones.drop(columns=[col], inplace=True)

# Step 3 — merge
top_zones = top_zones.merge(rc_agg[["h3_cell", "road_class", "lanes", "v0_kmh"]], on="h3_cell", how="left")
top_zones["road_class"] = top_zones["road_class"].fillna("collector")
top_zones["lanes"]      = top_zones["lanes"].fillna(2).astype(int)
top_zones["v0_kmh"]     = top_zones["v0_kmh"].fillna(40).astype(int)

print(rc_agg["road_class"].value_counts())
print(top_zones[["h3_cell", "road_class", "lanes", "v0_kmh"]].head(10).to_string())

road_class
arterial     1564
collector     511
local         459
Name: count, dtype: int64
           h3_cell road_class  lanes  v0_kmh
0  8960145ac57ffff      local      1      30
1  8961892f463ffff      local      1      30
2  8961892f66bffff   arterial      2      50
3  8961892f5a7ffff      local      1      30
4  8960145141bffff      local      1      30
5  89618926b8bffff   arterial      2      50
6  8961892ab8fffff      local      1      30
7  8961892e417ffff      local      1      30
8  8961892f647ffff   arterial      2      50
9  896189209bbffff      local      1      30


In [ ]:
# Cell 7 REPLACEMENT — V from road class (ETA API access denied, documented fallback)
# V assumed at 60% saturation for arterial, 40% for others — conservative urban estimate

ROAD_CLASS_VC = {
    "motorway":  0.70,
    "arterial":  0.60,
    "collector": 0.45,
    "local":     0.35,
    "unknown":   0.45,
}

top_zones["eta_ratio"]     = 1.0   # not available
top_zones["vc_calibrated"] = top_zones["road_class"].map(ROAD_CLASS_VC).fillna(0.45)
top_zones["C"]             = top_zones["lanes"] * SAT_FLOW
top_zones["V_calibrated"]  = (top_zones["vc_calibrated"] * top_zones["C"]).round(0)

print("V calibrated from road class (ETA API restricted — documented assumption):")
print(top_zones[["h3_cell", "road_class", "lanes", "C", "vc_calibrated", "V_calibrated"]].head(10).to_string())

V calibrated from road class (ETA API restricted — documented assumption):
           h3_cell road_class  lanes     C  vc_calibrated  V_calibrated
0  8960145ac57ffff      local      1  1800           0.35         630.0
1  8961892f463ffff      local      1  1800           0.35         630.0
2  8961892f66bffff   arterial      2  3600           0.60        2160.0
3  8961892f5a7ffff      local      1  1800           0.35         630.0
4  8960145141bffff      local      1  1800           0.35         630.0
5  89618926b8bffff   arterial      2  3600           0.60        2160.0
6  8961892ab8fffff      local      1  1800           0.35         630.0
7  8961892e417ffff      local      1  1800           0.35         630.0
8  8961892f647ffff   arterial      2  3600           0.60        2160.0
9  896189209bbffff      local      1  1800           0.35         630.0


In [ ]:
# ── CELL 8: MBPR Impact Engine ───────────────────────────────────────────────
# Reference: Gore, Arkatkar, Joshi & Antoniou (2023), TRR 2677(5):966-990 [form]
#            Anwar, Fujiwara & Zhang (2011) [alpha=3.59, beta=0.40]

def footprint_to_cio_fraction(vehicle_footprint):
    """Map average vehicle footprint to C_io fraction (capacity lost per lane)."""
    # Find nearest key
    keys = sorted(FOOTPRINT_C_IO.keys())
    for k in keys:
        if vehicle_footprint <= k:
            return FOOTPRINT_C_IO[k]
    return FOOTPRINT_C_IO[keys[-1]]

def compute_impact(row, window_h=ENFORCEMENT_WINDOW_H):
    """
    Compute delay_ratio, veh_min_lost, rupees_lost for one hotspot zone.
    Uses MBPR: T_a = T_0 * [1 + alpha*(V/(C - C_io))^beta]
    """
    lanes      = row.get("lanes", 2)
    v0_kmh     = row.get("v0_kmh", 40)
    C          = lanes * SAT_FLOW
    V          = row.get("V_calibrated", 0.3 * C)   # fallback: 30% saturation

    # Severity-weighted footprint → C_io
    avg_fp         = row.get("vehicle_footprint", 1.0)
    severity       = row.get("severity_score", 1.0)
    cio_frac       = footprint_to_cio_fraction(avg_fp)
    # Severity multiplier: high-severity (score>=5) blocks more capacity
    sev_mult       = min(severity / 5.0, 1.0)
    C_io           = cio_frac * sev_mult * SAT_FLOW   # capacity lost (one lane equivalent)
    C_effective    = max(C - C_io, 0.1 * C)           # floor at 10% capacity

    # Free-flow travel time (T_0): assume a 500 m reference segment
    seg_km = 0.5
    T_0    = (seg_km / v0_kmh) * 60   # minutes

    # BPR: T_a with illegal occupancy
    vc_with    = V / C_effective
    T_a_with   = T_0 * (1 + ALPHA * (vc_with ** BETA))

    # BPR: T_a without illegal occupancy (baseline)
    vc_without = V / C
    T_a_without = T_0 * (1 + ALPHA * (vc_without ** BETA))

    delay_ratio     = T_a_with / T_a_without if T_a_without > 0 else 1.0
    delay_per_veh   = T_a_with - T_a_without   # minutes per vehicle

    # Vehicle-minutes lost per day
    window_min      = window_h * 60
    # Vehicles passing = V * window_h
    veh_passing     = V * window_h
    veh_min_lost    = delay_per_veh * veh_passing

    # Rupees lost
    rupees_lost     = veh_min_lost * VOT_RS_PER_MIN

    return {
        "delay_ratio":   round(delay_ratio, 4),
        "veh_min_lost":  round(veh_min_lost, 1),
        "rupees_lost":   round(rupees_lost, 0),
        "C_io":          round(C_io, 1),
    }

# Apply to top zones
# Need vehicle_footprint — get from clean_violations parquet aggregated per cell
viol = pd.read_parquet(f"{DATA_DIR}/clean_violations.parquet", columns=["h3_cell", "vehicle_footprint", "severity_score"])
fp_by_cell = viol.groupby("h3_cell").agg(
    avg_footprint=("vehicle_footprint", "mean"),
).reset_index()

top_zones = top_zones.merge(fp_by_cell, on="h3_cell", how="left")
top_zones["vehicle_footprint"] = top_zones["avg_footprint"].fillna(1.0)

impact_results = top_zones.apply(compute_impact, axis=1, result_type="expand")
top_zones = pd.concat([top_zones, impact_results], axis=1)

print("MBPR impact computed:")
print(top_zones[["h3_cell", "delay_ratio", "veh_min_lost", "rupees_lost", "V_calibrated", "C_io"]].head(10).to_string())

MBPR impact computed:
           h3_cell  delay_ratio  veh_min_lost  rupees_lost  V_calibrated   C_io
0  8960145ac57ffff       1.0302         511.8       1535.0         630.0  180.0
1  8961892f463ffff       1.0302         511.8       1535.0         630.0  180.0
2  8961892f66bffff       1.0409        1664.7       4994.0        2160.0  450.0
3  8961892f5a7ffff       1.0856        1449.9       4350.0         630.0  450.0
4  8960145141bffff       1.0238         403.2       1210.0         630.0  144.0
5  89618926b8bffff       1.0409        1664.7       4994.0        2160.0  450.0
6  8961892ab8fffff       1.0238         403.2       1210.0         630.0  144.0
7  8961892e417ffff       1.0238         403.2       1210.0         630.0  144.0
8  8961892f647ffff       1.0409        1664.7       4994.0        2160.0  450.0
9  896189209bbffff       1.0238         403.2       1210.0         630.0  144.0


In [ ]:
# ── CELL 9: Impact-weighted rank ─────────────────────────────────────────────
# impact_weighted_rank = risk_debiased × veh_min_lost × severity_score
# (as per battle plan §3 P-ROI)

top_zones["impact_weighted_rank"] = (
    top_zones["risk_debiased"]
    * top_zones["veh_min_lost"].clip(lower=0)
    * top_zones["severity_score"].clip(lower=0.1)
)

# Rank
top_zones["rank"] = top_zones["impact_weighted_rank"].rank(ascending=False).astype(int)
top_zones.sort_values("rank", inplace=True)

print("Top 10 impact-ranked zones:")
cols = ["rank", "h3_cell", "lat", "lon", "blindspot_flag", "poi_demand",
        "delay_ratio", "veh_min_lost", "rupees_lost", "impact_weighted_rank"]
print(top_zones[cols].head(10).to_string())

Top 10 impact-ranked zones:
    rank          h3_cell        lat        lon  blindspot_flag  poi_demand  delay_ratio  veh_min_lost  rupees_lost  impact_weighted_rank
2      1  8961892f66bffff  13.063528  77.740034               1           5       1.0409        1664.7       4994.0            143.133633
3      2  8961892f5a7ffff  13.014433  77.710251               1          13       1.0856        1449.9       4350.0             84.304645
5      3  89618926b8bffff  12.847023  77.588790               1           5       1.0409        1664.7       4994.0             79.744625
8      4  8961892f647ffff  13.059233  77.744783               1           5       1.0409        1664.7       4994.0             69.884191
17     5  8961892e4c3ffff  12.978044  77.716856               1          14       1.0409        1664.7       4994.0             62.767609
20     6  89618925acfffff  12.965593  77.613050               1          13       1.0409        1664.7       4994.0             51.368664
0     

In [ ]:
# ── CELL 10: Merge enriched fields back to full hotspots → hotspots_enriched.csv
enrich_cols = ["h3_cell", "poi_demand", "lanes", "v0_kmh", "road_class",
               "V_calibrated", "delay_ratio", "veh_min_lost", "rupees_lost",
               "impact_weighted_rank", "eta_ratio", "C_io"]

# hotspots already has poi_demand from CELL 5
# merge remaining BPR fields
bpr_cols = [c for c in enrich_cols if c not in ["h3_cell", "poi_demand"]]
hotspots_enriched = hotspots.merge(
    top_zones[["h3_cell"] + bpr_cols],
    on="h3_cell", how="left"
)

# Fill NaN for non-enriched cells
for c in bpr_cols:
    if c in hotspots_enriched.columns:
        hotspots_enriched[c] = hotspots_enriched[c].fillna(0)

out_path = f"{OUT_DIR}/hotspots_enriched.csv"
hotspots_enriched.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Columns: {hotspots_enriched.columns.tolist()}")
print(f"Shape:   {hotspots_enriched.shape}")

Saved: /content/output/hotspots_enriched.csv
Columns: ['h3_cell', 'risk_debiased', 'violation_count', 'exposure', 'severity_score', 'peak_dow', 'blindspot_flag', 'lat', 'lon', 'poi_demand', 'is_treated', 'deterrence_effect', 'lanes', 'v0_kmh', 'road_class', 'V_calibrated', 'delay_ratio', 'veh_min_lost', 'rupees_lost', 'impact_weighted_rank', 'eta_ratio', 'C_io']
Shape:   (2534, 22)


In [ ]:
# ── CELL 11: Fair VRP Deployment Plan ────────────────────────────────────────
# Rank zones per police_station by impact_weighted_rank.
# Allocate N officers to top zones, with Gini fairness check.
#
# Note: We use a simple greedy VRP here (no Mappls Route Optimization quota needed).
# If you have Mappls VRPP quota, replace _greedy_vrp with a Mappls API call.

# Load clean_violations to get police_station per h3_cell
viol2 = pd.read_parquet(
    f"{DATA_DIR}/clean_violations.parquet",
    columns=["h3_cell", "police_station"]
)
ps_map = viol2.groupby("h3_cell")["police_station"].agg(lambda x: x.mode()[0]).reset_index()

# Add police_station to top_zones
top_zones_ps = top_zones.merge(ps_map, on="h3_cell", how="left")
top_zones_ps["police_station"] = top_zones_ps["police_station"].fillna("Unknown")

OFFICERS_PER_STATION = 3   # change as needed

def gini(arr):
    """Gini coefficient (0=perfect equality, 1=total inequality)."""
    arr = np.array(arr, dtype=float)
    if arr.sum() == 0:
        return 0.0
    arr.sort()
    n = len(arr)
    idx = np.arange(1, n+1)
    return (2 * np.sum(idx * arr) / (n * arr.sum())) - (n+1)/n

deployment_plan = []
for station, grp in top_zones_ps.groupby("police_station"):
    grp_sorted = grp.sort_values("impact_weighted_rank", ascending=False)
    allocated  = grp_sorted.head(OFFICERS_PER_STATION)

    for i, (_, zone) in enumerate(allocated.iterrows()):
        deployment_plan.append({
            "police_station":      station,
            "officer_slot":        i + 1,
            "h3_cell":             zone["h3_cell"],
            "lat":                 zone["lat"],
            "lon":                 zone["lon"],
            "blindspot_flag":      int(zone["blindspot_flag"]),
            "poi_demand":          int(zone.get("poi_demand", 0)),
            "impact_weighted_rank": zone.get("impact_weighted_rank", 0),
            "veh_min_lost":        zone.get("veh_min_lost", 0),
            "rupees_lost":         zone.get("rupees_lost", 0),
            "severity_score":      zone["severity_score"],
            "peak_dow":            int(zone["peak_dow"]),
        })

deploy_df = pd.DataFrame(deployment_plan)

# Gini on veh_min_lost allocation per station
station_load = deploy_df.groupby("police_station")["veh_min_lost"].sum()
fairness_gini = gini(station_load.values)
print(f"Deployment fairness Gini (lower=better): {fairness_gini:.3f}")
print(f"Stations covered: {deploy_df['police_station'].nunique()}")
print(f"Total patrol assignments: {len(deploy_df)}")

# Save
routing_out = {
    "fairness_gini":     fairness_gini,
    "officers_per_station": OFFICERS_PER_STATION,
    "plan":              deploy_df.to_dict(orient="records")
}
with open(f"{OUT_DIR}/routing_plan.json", "w") as f:
    json.dump(routing_out, f, indent=2)
print(f"Saved: {OUT_DIR}/routing_plan.json")

deploy_df.to_csv(f"{OUT_DIR}/routing_plan.csv", index=False)
print(f"Saved: {OUT_DIR}/routing_plan.csv")

print("\nSample plan:")
print(deploy_df.head(10).to_string())

Deployment fairness Gini (lower=better): 0.604
Stations covered: 33
Total patrol assignments: 70
Saved: /content/output/routing_plan.json
Saved: /content/output/routing_plan.csv

Sample plan:
  police_station  officer_slot          h3_cell        lat        lon  blindspot_flag  poi_demand  impact_weighted_rank  veh_min_lost  rupees_lost  severity_score  peak_dow
0        Adugodi             1  89618925cc3ffff  12.925985  77.627429               1          14              0.417772          96.5        289.0             1.0         1
1    Ashok Nagar             1  89618925acfffff  12.965593  77.613050               1          13             51.368664        1664.7       4994.0             6.0         1
2   Banashankari             1  8960145a01bffff  12.942309  77.546476               1          15             36.244546        1664.7       4994.0             5.0         3
3   Banashankari             2  896189259afffff  12.928318  77.574513               1          14              9.496

In [ ]:
# ── CELL 12: impact_bpr.recompute() — interface for Member C's twin ──────────
# Member C calls this signature: impact_bpr.recompute(segment_id, c_io) -> dict
# We save this as a module file for C to import.

module_code = '''
# impact_bpr.py — Member B exposes this for Member C twin
# Generated by PARKLENS Member B notebook

ALPHA = 3.59
BETA  = 0.40
SAT_FLOW = 1800
VOT_RS_PER_MIN = 3.0

ROAD_CLASS_DEFAULTS = {
    "motorway":  {"lanes": 3, "v0_kmh": 80},
    "arterial":  {"lanes": 2, "v0_kmh": 50},
    "collector": {"lanes": 2, "v0_kmh": 40},
    "local":     {"lanes": 1, "v0_kmh": 30},
    "unknown":   {"lanes": 2, "v0_kmh": 40},
}

_segment_registry = {}   # loaded from hotspots_enriched.csv at startup

def load_segments(hotspots_enriched_path):
    """Call once at startup to populate segment registry."""
    import pandas as pd
    global _segment_registry
    df = pd.read_csv(hotspots_enriched_path)
    for _, row in df.iterrows():
        rc   = row.get("road_class", "unknown")
        info = ROAD_CLASS_DEFAULTS.get(rc, ROAD_CLASS_DEFAULTS["unknown"])
        _segment_registry[row["h3_cell"]] = {
            "lanes":       row.get("lanes", info["lanes"]),
            "v0_kmh":      row.get("v0_kmh", info["v0_kmh"]),
            "V_calibrated": row.get("V_calibrated", 0.3 * info["lanes"] * SAT_FLOW),
            "road_class":  rc,
        }

def recompute(segment_id: str, c_io: float) -> dict:
    """
    Recompute BPR delay for a segment with given C_io (capacity removed).
    Returns: {T_a_with, T_a_without, delay_ratio, veh_min_lost, rupees_lost}
    """
    seg = _segment_registry.get(segment_id, {})
    lanes  = seg.get("lanes", 2)
    v0_kmh = seg.get("v0_kmh", 40)
    V      = seg.get("V_calibrated", 0.3 * lanes * SAT_FLOW)
    C      = lanes * SAT_FLOW

    C_eff = max(C - c_io, 0.1 * C)
    seg_km = 0.5
    T_0    = (seg_km / v0_kmh) * 60

    T_a_with    = T_0 * (1 + ALPHA * (V / C_eff) ** BETA)
    T_a_without = T_0 * (1 + ALPHA * (V / C)    ** BETA)

    delay_ratio   = T_a_with / T_a_without if T_a_without > 0 else 1.0
    delay_per_veh = T_a_with - T_a_without
    veh_min_lost  = delay_per_veh * V * 8   # 8h window
    rupees_lost   = veh_min_lost * VOT_RS_PER_MIN

    return {
        "T_a_with":     round(T_a_with, 4),
        "T_a_without":  round(T_a_without, 4),
        "delay_ratio":  round(delay_ratio, 4),
        "veh_min_lost": round(veh_min_lost, 1),
        "rupees_lost":  round(rupees_lost, 0),
    }
'''

with open(f"{OUT_DIR}/impact_bpr.py", "w") as f:
    f.write(module_code)
print(f"Saved: {OUT_DIR}/impact_bpr.py")
print("Member C: import impact_bpr; impact_bpr.load_segments('hotspots_enriched.csv'); impact_bpr.recompute(h3_cell, c_io)")

Saved: /content/output/impact_bpr.py
Member C: import impact_bpr; impact_bpr.load_segments('hotspots_enriched.csv'); impact_bpr.recompute(h3_cell, c_io)


In [ ]:
# ── CELL 13: Save mappls_client.py for Member C ───────────────────────────────
client_code = f'''
# mappls_client.py — shared Mappls client (Member B → Member A & C)
# Set MAPPLS_KEY before importing

import json, time, requests
from pathlib import Path

MAPPLS_KEY = "{MAPPLS_KEY}"   # filled by Member B
CACHE_FILE = Path("mappls_cache.json")

def _load(): return json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {{}}
def _save(c): CACHE_FILE.write_text(json.dumps(c, indent=2))
CACHE = _load()

def route_geometry(origin_lat, origin_lon, dest_lat, dest_lon, corridor_id=None):
    key = f"route_{{corridor_id or f{{origin_lat:.4f}}_{{origin_lon:.4f}}}}"
    if key in CACHE: return CACHE[key]
    url = (
        f"https://apis.mappls.com/advancedmaps/v1/{{MAPPLS_KEY}}/route_adv/driving"
        f"/{{origin_lon}},{{origin_lat}};{{dest_lon}},{{dest_lat}}"
        f"?geometries=geojson&steps=false"
    )
    result = {{"type": "LineString", "coordinates": []}}
    try:
        r = requests.get(url, timeout=15)
        result = r.json()["routes"][0]["geometry"]
    except: pass
    CACHE[key] = result; _save(CACHE); time.sleep(0.2)
    return result
'''
with open(f"{OUT_DIR}/mappls_client.py", "w") as f:
    f.write(client_code)
print(f"Saved: {OUT_DIR}/mappls_client.py")

Saved: /content/output/mappls_client.py


In [ ]:
# ── CELL 14: Summary + sanity checks ─────────────────────────────────────────
import os

print("=" * 60)
print("MEMBER B DELIVERABLES")
print("=" * 60)

outputs = [
    "hotspots_enriched.csv",
    "routing_plan.csv",
    "routing_plan.json",
    "mappls_cache.json",
    "impact_bpr.py",
    "mappls_client.py",
]
for f in outputs:
    path = f"{OUT_DIR}/{f}"
    size = os.path.getsize(path) if os.path.exists(path) else 0
    print(f"  {'✓' if size>0 else '✗'}  {f:35s} {size/1024:.1f} KB")

print()
print("TOP 5 IMPACT ZONES (hand to Member C for twin demo):")
twin_cols = ["rank", "h3_cell", "lat", "lon", "blindspot_flag", "poi_demand",
             "delay_ratio", "veh_min_lost", "rupees_lost"]
print(top_zones[twin_cols].head(5).to_string())

print()
print(f"Deployment fairness (Gini): {fairness_gini:.3f} (0=fair, 1=unfair)")
print(f"Cache entries saved:        {len(CACHE)}")
print()
print("Hand-off to Member C:")
print("  import impact_bpr")
print("  impact_bpr.load_segments('/content/output/hotspots_enriched.csv')")
print("  impact_bpr.recompute('<h3_cell>', c_io=450)")
print()
print("Judge-ready citation:")
print("  MBPR form: Gore et al. (2023), TRR 2677(5):966-990")
print("  Constants: Anwar et al. (2011), Dhaka alpha=3.59 beta=0.40")
print(" V calibrated from violation-weighted road class inference (avg vehicle footprint + violation density per H3 cell).")

MEMBER B DELIVERABLES
  ✓  hotspots_enriched.csv               352.0 KB
  ✓  routing_plan.csv                    7.6 KB
  ✓  routing_plan.json                   26.4 KB
  ✓  mappls_cache.json                   2.9 KB
  ✓  impact_bpr.py                       2.2 KB
  ✓  mappls_client.py                    1.1 KB

TOP 5 IMPACT ZONES (hand to Member C for twin demo):
    rank          h3_cell        lat        lon  blindspot_flag  poi_demand  delay_ratio  veh_min_lost  rupees_lost
2      1  8961892f66bffff  13.063528  77.740034               1           5       1.0409        1664.7       4994.0
3      2  8961892f5a7ffff  13.014433  77.710251               1          13       1.0856        1449.9       4350.0
5      3  89618926b8bffff  12.847023  77.588790               1           5       1.0409        1664.7       4994.0
8      4  8961892f647ffff  13.059233  77.744783               1           5       1.0409        1664.7       4994.0
17     5  8961892e4c3ffff  12.978044  77.716856      

In [ ]:
print(rc_agg["road_class"].value_counts())
print(rc_agg[["avg_footprint", "total_violations"]].describe())

road_class
arterial     1564
collector     511
local         459
Name: count, dtype: int64
       avg_footprint  total_violations
count    2534.000000       2534.000000
mean        0.885217        117.651934
std         0.363514        505.866823
min         0.300000          1.000000
25%         0.663158          3.000000
50%         0.906066         11.000000
75%         1.000000         54.000000
max         3.000000      12106.000000


In [ ]:
print(top_zones["road_class"].value_counts())

road_class
local        80
arterial     16
collector     4
Name: count, dtype: int64


In [ ]:
import requests

API_KEY = "cnwthsmvduobihdgbkwdaqxjhglhxpiqamca"

url = (
    "https://route.mappls.com/route/dm/distance_matrix/driving/"
    "77.983936,28.255904;"
    "77.05993,28.487555;"
    "77.15993,28.587555;"
    "17ZUL7"
)

params = {
    "rtype": 0,
    "region": "ind",
    "access_token": API_KEY
}

response = requests.get(url, params=params)

print("Status:", response.status_code)
print(response.text)

Status: 200
{"Server":"DE-5400","version":"202404.221.5223","results":{"distances":[[0,139332,120613.9,112976.7]],"code":"Ok","durations":[[0,8541.2,7339.3,6736.3]]},"responseCode":200}


In [ ]:
import requests
import json

API_KEY = "cnwthsmvduobihdgbkwdaqxjhglhxpiqamca"

url = f"https://search.mappls.com/search/places/geo-location?access_token={API_KEY}"

payload = {
    "radioType": "lte",
    "cellTowers": [
        {
            "cellId": 900372,
            "locationAreaCode": 57,
            "mobileCountryCode": 405,
            "mobileNetworkCode": 872
        }
    ]
}

response = requests.post(
    url,
    data=json.dumps(payload),
    headers={"Content-Type": "application/json"}
)

print(response.status_code)
print(response.text)

401



In [ ]:
import requests

API_KEY = "cnwthsmvduobihdgbkwdaqxjhglhxpiqamca"

url = "https://route.mappls.com/routev2/dm/distance"

params = {
    "source": "77.26893112158996,28.550614332693453",
    "target": "77.20815617692985,28.566618140144396",
    "profile": "driving",
    "speedTypes": "predictive",
    "date_time": "1,2021-12-20T11:00",
    "access_token": API_KEY
}

response = requests.get(url, params=params)

print("Final URL:", response.url)
print("Status:", response.status_code)
print(response.text)

Final URL: https://route.mappls.com/routev2/dm/distance?source=77.26893112158996%2C28.550614332693453&target=77.20815617692985%2C28.566618140144396&profile=driving&speedTypes=predictive&date_time=1%2C2021-12-20T11%3A00&access_token=cnwthsmvduobihdgbkwdaqxjhglhxpiqamca
Status: 200
{"responsecode":401,"error_description":"Api access denied : app1781834446301i640557321 , ast1621577442i396742397","error_code":"ASSET_ACCESS_DENIED","error":"Api Access Denied"}


In [ ]:
import requests

API_KEY = ""

url = f"https://route.mappls.com/routev2/movement/trace_route?access_token={API_KEY}"

payload = {
    "points": "72.8632,21.207756;72.86322,21.207582;72.86325,21.207506;72.86324,21.207438;72.86321,21.207426",
    "type": "break",
    "search_radius": 12
}

headers = {
    "Content-Type": "application/x-www-form-urlencoded"
}

response = requests.post(url, data=payload, headers=headers)

print("Status:", response.status_code)
print(response.text)

Status: 200
{"responsecode":401,"error_description":"Api access denied : app1781834446301i640557321 , ast1704258395i892011725","error_code":"ASSET_ACCESS_DENIED","error":"Api Access Denied"}


In [ ]:
import os
import requests

# ← paste your Mappls key here
MAPPLS_KEY = "cnwthsmvduobihdgbkwdaqxjhglhxpiqamca"

P1 = "77.5946,12.9716"
P2 = "77.6200,12.9400"
TIMEOUT = 20

def _verdict(name, ok, detail):
    mark = "✅ PASS" if ok else "❌ FAIL"
    print(f"{mark}  {name}")
    print(f"        {detail}\n")

def check_distance_matrix(key):
    url = (f"https://route.mappls.com/route/dm/distance_matrix/driving/"
           f"{P1};{P2}?access_token={key}")
    try:
        r = requests.get(url, timeout=TIMEOUT)
        data = r.json()
        durs = data.get("results", {}).get("durations")
        if r.status_code == 200 and durs:
            secs = durs[0][1]
            return _verdict("Distance Matrix (ETA)", True,
                            f"Travel time returned: ~{secs/60:.1f} min. ETA calibration available.")
        return _verdict("Distance Matrix (ETA)", False,
                        f"HTTP {r.status_code}, responseCode {data.get('responseCode')}. Body: {str(data)[:160]}")
    except Exception as e:
        return _verdict("Distance Matrix (ETA)", False, f"Request failed: {e}")

def check_directions(key):
    url = (f"https://route.mappls.com/route/direction/route_adv/driving/"
           f"{P1};{P2}?access_token={key}")
    try:
        r = requests.get(url, timeout=TIMEOUT)
        data = r.json()
        routes = data.get("routes") or []
        if r.status_code == 200 and routes:
            dur = routes[0].get("duration")
            return _verdict("Directions / Routing", True,
                            f"Route returned with duration ~{(dur or 0)/60:.1f} min.")
        return _verdict("Directions / Routing", False,
                        f"HTTP {r.status_code}, responseCode {data.get('responseCode')}. Body: {str(data)[:160]}")
    except Exception as e:
        return _verdict("Directions / Routing", False, f"Request failed: {e}")

def check_snap_to_road(key):
    url = f"https://route.mappls.com/route/movement/snapToRoad?access_token={key}"
    try:
        r = requests.post(url, data={"pts": f"{P1};{P2}", "geometries": "polyline"}, timeout=TIMEOUT)
        data = r.json()
        snapped = data.get("results", {}).get("snappedPoints") or data.get("tracepoints")
        if r.status_code == 200 and snapped:
            return _verdict("Snap to Road", True, "Points snapped. Per-cell road lookup available.")
        return _verdict("Snap to Road", False,
                        f"HTTP {r.status_code}, responseCode {data.get('responseCode')}. Body: {str(data)[:160]}")
    except Exception as e:
        return _verdict("Snap to Road", False, f"Request failed: {e}")

# --- run ---
print(f"Testing Mappls key ...{MAPPLS_KEY[-4:]}  (2 sample points in Bengaluru)\n")
check_distance_matrix(MAPPLS_KEY)
check_directions(MAPPLS_KEY)
check_snap_to_road(MAPPLS_KEY)
print("If Distance Matrix is ✅ you can calibrate traffic volume from real ETAs.")
print("If anything is ❌, enable that API in Mappls Console → your app, then re-run.")
r = requests.post(
    f"https://route.mappls.com/route/movement/snapToRoad?access_token={MAPPLS_KEY}",
    data={"pts": "77.5946,12.9716;77.6200,12.9400", "geometries": "polyline"},
    timeout=10
)
print(r.status_code)
print(r.text[:1000])

Testing Mappls key ...amca  (2 sample points in Bengaluru)

✅ PASS  Distance Matrix (ETA)
        Travel time returned: ~12.2 min. ETA calibration available.

✅ PASS  Directions / Routing
        Route returned with duration ~12.2 min.

✅ PASS  Snap to Road
        Points snapped. Per-cell road lookup available.

If Distance Matrix is ✅ you can calibrate traffic volume from real ETAs.
If anything is ❌, enable that API in Mappls Console → your app, then re-run.
200
{"Server":"DE-5402","version":"202404.221.5223","results":{"snappedPoints":[{"distance":6.431319,"location":[77.594542,12.971612],"waypoint_index":0},{"distance":4.786495,"location":[77.619964,12.940025],"waypoint_index":1}]},"responseCode":200}
